# Pipeline HCM — phiên bản A1 (chỉ LightGBM)

A1 dựng trên pipeline của công ty (**phiên bản A**, `A/pipeline_HCM_A.ipynb`), chỉ dùng **LightGBM**, đầu vào là dữ liệu đã làm sạch ở `output/`.

**Khác biệt so với A**

| | A | A1 |
|---|---|---|
| Dữ liệu | `train_HCM_3.xlsx`, `tstd_HCM_3.xlsx` gốc | `output/train_HCM_3_clean.xlsx`, `output/tstd_HCM_3_clean.xlsx` |
| Mô hình | LightGBM + CatBoost | chỉ LightGBM |
| Biến tính từ đơn giá | `Don_gia_TB`, `Min`, `Max` (LOO khi train, đọc từ file khi test), 5 cột `DG_*` | **không dùng** cột nào tính từ đơn giá: `Don_gia_TB`, `Min`, `Max`, `Độ lệch chuẩn`, `Số lượng mẫu`, `DG_*`; không còn bộ mã hoá LOO |
| Cột chỉ số | `Chi_so_dientich`, `Chi_so_ngo`, `Chi_so_hinhdang` | **không dùng** |
| Hình dáng | qua `Chi_so_hinhdang` | `hinh_dang_sach` (1 cột phân loại, 45 nhãn) |
| `Đường phố1`, `Phan_loai_VT` | có | **bỏ** (dữ liệu sạch không có; `Phan_loai_VT` trùng thông tin với VT) |
| Cờ True/False của bước làm sạch | — | **không dùng** (`la_hem`, `co_mat_tien`, `canh_bao_*`…) |
| Cột tỷ lệ `ty_le_kt_dt`, `ty_le_dai_rong` | — | **không dùng** |
| Đánh giá | Train / Val / Test | như A **+ tách theo vùng HCM1 / HCM2** |

Phần còn lại giữ như A: chia train/val ngẫu nhiên 80/20 (`SEED = 42`), target `log1p`, cùng tham số LightGBM và early stopping 200 vòng, cùng bộ mã hoá `Chi tiết` (chuẩn hoá + 9 cờ + TF-IDF → SVD 24 chiều), cùng TF-IDF cho `Yếu tố khác`, biến số thiếu điền bằng trung vị tập train, cùng các chỉ số đánh giá.

## 1. Cấu hình

In [1]:
import os, re, json, time, hashlib, unicodedata, warnings, platform
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import joblib, sklearn, scipy
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

# Tự tìm thư mục gốc dự án (thư mục có data/processed/train_HCM_3_clean.xlsx), mở notebook từ đâu cũng chạy được
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "processed" / "train_HCM_3_clean.xlsx").exists())
VERSION   = "HCM_A1"
TRAIN_XLS = ROOT / "data" / "processed" / "train_HCM_3_clean.xlsx"
TEST_XLS  = ROOT / "data" / "processed" / "tstd_HCM_3_clean.xlsx"
A1_DIR    = ROOT / "02_mo_hinh" / "LightGBM"
MODEL_DIR = A1_DIR / "model"
RES_DIR   = A1_DIR / "ket_qua"
os.makedirs(RES_DIR, exist_ok=True)

SEED, TEST_SIZE = 42, 0.2
THREADS = os.cpu_count()                # A đặt 20 theo máy công ty; A1 dùng số luồng của máy đang chạy
CHITIET_SVD_DIM = 24

LGB_PARAMS = dict(objective="regression", learning_rate=0.03, n_estimators=12000,
                  num_leaves=255, min_child_samples=5, feature_fraction=0.8,
                  bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  num_threads=THREADS, random_state=SEED, verbose=-1)
EARLY_STOP = 200
print("ROOT:", ROOT, "\nMODEL_DIR:", MODEL_DIR, "\nRES_DIR:", RES_DIR)
print("versions: python", platform.python_version(), "| pandas", pd.__version__,
      "| lightgbm", lgb.__version__, "| sklearn", sklearn.__version__)

ROOT: D:\HCM2810 
MODEL_DIR: D:\HCM2810\A1\model_HCM_A1 
RES_DIR: D:\HCM2810\A1\ket_qua
versions: python 3.14.4 | pandas 3.0.5 | lightgbm 4.7.0 | sklearn 1.9.0


## 2. Khai báo cột

So với A: bỏ `Đường phố1`, `Phan_loai_VT`, ba cột `Chi_so_*` và toàn bộ nhóm thống kê đơn giá; thêm `hinh_dang_sach` vào nhóm phân loại.

In [2]:
TARGET = "Đơn giá quyền sử dụng đất (đ/m2)1"
XA     = "Xã/Phường/Thị trấn mới"
VUNG   = "Vùng"                              # hcm1 / hcm2 (dữ liệu sạch đã viết thường)

CAT_BASE = [XA, VUNG, "Đường phố", "Vị trí trong khung giá", "Quý_Năm",
            "Lợi thế kinh doanh", "hinh_dang_sach"]
NUM_BASE = ["Diện tích (m2)", "Kích thước mặt tiền (m)", "Kích thước chiều dài",
            "Số mặt tiền tiếp giáp", "Độ rộng ngõ/ngách nhỏ nhất (Từ đường chính đến BĐS)",
            "Khoảng cách đến đường chính (m)"]
TEXT_TFIDF = ["Yếu tố khác"]
CHITIET    = "Chi tiết"

BASE_ORDER = [XA, VUNG, "Đường phố",
              "Diện tích (m2)", "Kích thước mặt tiền (m)", "Kích thước chiều dài",
              "Số mặt tiền tiếp giáp", "Độ rộng ngõ/ngách nhỏ nhất (Từ đường chính đến BĐS)",
              "Khoảng cách đến đường chính (m)", "Vị trí trong khung giá",
              "Lợi thế kinh doanh", "hinh_dang_sach", "Quý_Năm"]

# Cột có trong dữ liệu sạch nhưng A1 CỐ Ý không cho mô hình học
EXCLUDED = {
    "tính từ đơn giá":       ["Don_gia_TB", "Min", "Max", "Độ lệch chuẩn", "Số lượng mẫu"],
    "chỉ số":                ["Chi_so_ngo", "Chi_so_dientich", "Chi_so_hinhdang"],
    "cờ True/False":         ["la_hem", "co_mat_tien", "chi_la_mat_tien", "la_so_nha",
                              "co_bang_chung_mat_tien", "da_sua_vt_theo_mat_tien",
                              "kc_thieu_du_lieu", "da_sua_kich_thuoc", "canh_bao_vt1",
                              "canh_bao_mt_vt234"],
    "thay bằng hinh_dang_sach": ["Hình dáng"],
    "tỷ lệ (chỉ để quan sát)":  ["ty_le_kt_dt", "ty_le_dai_rong"],
    "A cũng không dùng":     ["Thành phố/Quận/Huyện/Thị xã", "Ghep", "Đoạn đường", "Mã tài sản", "Tháng"],
}
EXCLUDED_ALL = [c for v in EXCLUDED.values() for c in v]
USED_RAW = CAT_BASE + NUM_BASE + TEXT_TFIDF + [CHITIET]
assert not set(USED_RAW) & set(EXCLUDED_ALL)
print(f"{len(CAT_BASE)} cột phân loại | {len(NUM_BASE)} cột số | TF-IDF: {TEXT_TFIDF} | embedding: {CHITIET}")
for k, v in EXCLUDED.items():
    print(f"  bỏ ({k}): {v}")

7 cột phân loại | 6 cột số | TF-IDF: ['Yếu tố khác'] | embedding: Chi tiết
  bỏ (tính từ đơn giá): ['Don_gia_TB', 'Min', 'Max', 'Độ lệch chuẩn', 'Số lượng mẫu']
  bỏ (chỉ số): ['Chi_so_ngo', 'Chi_so_dientich', 'Chi_so_hinhdang']
  bỏ (cờ True/False): ['la_hem', 'co_mat_tien', 'chi_la_mat_tien', 'la_so_nha', 'co_bang_chung_mat_tien', 'da_sua_vt_theo_mat_tien', 'kc_thieu_du_lieu', 'da_sua_kich_thuoc', 'canh_bao_vt1', 'canh_bao_mt_vt234']
  bỏ (thay bằng hinh_dang_sach): ['Hình dáng']
  bỏ (tỷ lệ (chỉ để quan sát)): ['ty_le_kt_dt', 'ty_le_dai_rong']
  bỏ (A cũng không dùng): ['Thành phố/Quận/Huyện/Thị xã', 'Ghep', 'Đoạn đường', 'Mã tài sản', 'Tháng']


## 3. Bộ mã hoá `Chi tiết`

Giữ nguyên bộ mã hoá của A (`norm_chitiet` + 9 cờ + TF-IDF từ 1–2 và ký tự 2–4 → TruncatedSVD). Bộ mã hoá giá LOO của A bị bỏ vì A1 không dùng biến tính từ đơn giá.

Lưu ý: `Chi tiết` trong dữ liệu sạch đã ở dạng các ý cách nhau bằng `; ` (`mt`, `hẻm`, `kdc …`). `norm_chitiet` bỏ dấu `;` và bung `mt` → `mặt tiền`, `kdc` → `khu dân cư`, nên các cờ `ct_*` vẫn bắt được.

In [3]:
_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_NUM_RE = re.compile(r"(\d+)\s*$")
_JUNK   = {"", ".", "..", "-", "/", "x", "na", "n a", "nan", "none", "0", "không", "khong"}
_ABBR   = {"kp": "khu phố", "kdc": "khu dân cư", "mt": "mặt tiền", "hxh": "hẻm xe hơi",
           "tđc": "tái định cư", "tdc": "tái định cư", "kcn": "khu công nghiệp",
           "qh": "quy hoạch", "kdt": "khu đô thị", "kđt": "khu đô thị", "cc": "chung cư",
           "hcc": "hẻm cụt", "đ": "đường", "p": "phường", "q": "quận"}

def norm_chitiet(s):
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFC", str(s)).replace("​", " ").strip().lower()
    if _URL_RE.search(s):
        return "<url>"
    s = re.sub(r"(?<=[a-zà-ỹ])(?=\d)", " ", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if s in _JUNK or len(s) <= 1:
        return ""
    return " ".join(_ABBR.get(t, t) for t in s.split()).strip()

def chitiet_flags(norm):
    f = pd.DataFrame(index=norm.index)
    f["ct_len"]      = norm.str.len()
    f["ct_missing"]  = (norm == "").astype(int)
    f["ct_hem"]      = norm.str.contains(r"hẻm|ngõ|ngách|hem", regex=True).astype(int)
    f["ct_mat_tien"] = norm.str.contains(r"mặt tiền|mat tien", regex=True).astype(int)
    f["ct_noi_bo"]   = norm.str.contains(r"nội bộ|noi bo", regex=True).astype(int)
    f["ct_nha"]      = norm.str.contains(r"\bnhà\b|biệt thự|chung cư", regex=True).astype(int)
    f["ct_dat"]      = norm.str.contains(r"\bđất\b", regex=True).astype(int)
    f["ct_du_an"]    = norm.str.contains(r"dự án|khu dân cư|khu đô thị|resort|villa", regex=True).astype(int)
    f["ct_num"]      = norm.str.extract(_NUM_RE)[0].astype(float).fillna(-1)
    return f

CT_FLAGS = ["ct_len", "ct_missing", "ct_hem", "ct_mat_tien", "ct_noi_bo",
            "ct_nha", "ct_dat", "ct_du_an", "ct_num"]

class ChiTietEncoder:
    """Chuẩn hoá + 9 cờ + TF-IDF (từ 1-2, ký tự char_wb 2-4) -> TruncatedSVD."""

    def __init__(self, n_components=CHITIET_SVD_DIM):
        self.n_components = n_components

    def fit(self, s):
        norm = s.map(norm_chitiet)
        self.vw = TfidfVectorizer(analyzer="word",    ngram_range=(1, 2), min_df=3, max_features=600)
        self.vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=5, max_features=1500)
        X = hstack([self.vw.fit_transform(norm), self.vc.fit_transform(norm)]).tocsr()
        self.dim = min(self.n_components, X.shape[1] - 1)
        self.svd = TruncatedSVD(n_components=self.dim, random_state=SEED).fit(X)
        return self

    def transform(self, s):
        norm = s.map(norm_chitiet)
        out = chitiet_flags(norm)
        E = self.svd.transform(hstack([self.vw.transform(norm), self.vc.transform(norm)]).tocsr())
        for i in range(self.dim):
            out[f"ct_emb_{i}"] = E[:, i]
        return out

    @property
    def columns(self):
        return CT_FLAGS + [f"ct_emb_{i}" for i in range(self.dim)]

    def get_state(self):
        return {"vw": self.vw, "vc": self.vc, "svd": self.svd,
                "dim": self.dim, "n_components": self.n_components}

    @classmethod
    def from_state(cls, s):
        o = cls(n_components=s["n_components"])
        o.vw, o.vc, o.svd, o.dim = s["vw"], s["vc"], s["svd"], s["dim"]
        return o

demo = pd.DataFrame({"chi tiết sạch": ["hẻm", "mt", "mt; hẻm", "hẻm; đường số 7; khu phố 5",
                                        "mt; đường nội bộ; kdc hai thành", np.nan]})
demo["chuẩn hoá"] = demo["chi tiết sạch"].map(norm_chitiet)
print(demo.to_string(index=False))

                  chi tiết sạch                                  chuẩn hoá
                            hẻm                                        hẻm
                             mt                                   mặt tiền
                        mt; hẻm                               mặt tiền hẻm
     hẻm; đường số 7; khu phố 5                   hẻm đường số 7 khu phố 5
mt; đường nội bộ; kdc hai thành mặt tiền đường nội bộ khu dân cư hai thành
                            NaN                                           


## 4. Nạp dữ liệu sạch & kiểm tra đầu vào

In [4]:
def load_clean(path, need_target=True):
    df = pd.read_excel(path)
    if need_target:
        df = df[df[TARGET].notna() & (df[TARGET] > 0)].reset_index(drop=True)
    return df

train_raw = load_clean(TRAIN_XLS)
test_raw  = load_clean(TEST_XLS)
y_train_all = train_raw[TARGET].astype(float).values
y_test      = test_raw[TARGET].astype(float).values
print("train", train_raw.shape, "| test", test_raw.shape)
print(f"giá train: trung vị {np.median(y_train_all):,.0f} đ/m² | test: {np.median(y_test):,.0f} đ/m²")

for name, df in [("train", train_raw), ("test", test_raw)]:
    miss = [c for c in USED_RAW if c not in df.columns]
    assert not miss, f"{name} thiếu cột: {miss}"
print("Đủ mọi cột A1 dùng ở cả train và test  ✓")

print("\nPhân bố vùng:")
print(pd.DataFrame({"train": train_raw[VUNG].value_counts(), "test": test_raw[VUNG].value_counts()}).to_string())
print("\nÔ thiếu ở các cột đầu vào (train | test):")
na = pd.DataFrame({"train": train_raw[USED_RAW].isna().sum(), "test": test_raw[USED_RAW].isna().sum()})
print(na[na.sum(axis=1) > 0].to_string())

_unseen = {c: int((~test_raw[c].isin(set(train_raw[c].dropna()))).sum()) for c in CAT_BASE}
print("\nGiá trị phân loại của test không có trong train (số dòng):", _unseen)

train (43466, 42) | test (693, 40)
giá train: trung vị 68,774,548 đ/m² | test: 70,943,321 đ/m²
Đủ mọi cột A1 dùng ở cả train và test  ✓

Phân bố vùng:
      train  test
Vùng             
hcm1  27869   435
hcm2  15597   258

Ô thiếu ở các cột đầu vào (train | test):
                                                     train  test
Đường phố                                               11     0
hinh_dang_sach                                          15     0
Kích thước mặt tiền (m)                                  1     0
Số mặt tiền tiếp giáp                                   21     0
Độ rộng ngõ/ngách nhỏ nhất (Từ đường chính đến BĐS)      3     0
Khoảng cách đến đường chính (m)                        737     1
Yếu tố khác                                            769     0
Chi tiết                                              4405    46



Giá trị phân loại của test không có trong train (số dòng): {'Xã/Phường/Thị trấn mới': 0, 'Vùng': 0, 'Đường phố': 0, 'Vị trí trong khung giá': 0, 'Quý_Năm': 0, 'Lợi thế kinh doanh': 0, 'hinh_dang_sach': 0}


## 5. Dựng đặc trưng

Khác A: không còn khối thống kê giá (`Don_gia_TB/Min/Max` + `DG_*`). Ô thiếu ở cột phân loại được gán `"Unknown"` (A gán chuỗi `"nan"` do `astype(str)` chạy trước `fillna`; hai cách đều gom ô thiếu thành một nhóm riêng).

In [5]:
def clean_base(df, medians=None):
    out = pd.DataFrame(index=df.index)
    for c in CAT_BASE:
        out[c] = df[c].astype(object).where(df[c].notna(), "Unknown").astype(str)
    for c in NUM_BASE:
        out[c] = pd.to_numeric(df[c], errors="coerce")
    if medians is None:
        medians = out[NUM_BASE].median()
    out[NUM_BASE] = out[NUM_BASE].fillna(medians)
    return out, medians

def fit_tfidf(df):
    return {c: TfidfVectorizer(max_features=600, stop_words="english",
                               ngram_range=(1, 2)).fit(df[c].fillna("").astype(str))
            for c in TEXT_TFIDF}

def apply_tfidf(vecs, df):
    blocks = []
    for col, v in vecs.items():
        A_ = v.transform(df[col].fillna("").astype(str)).toarray()
        pref = f'tfidf_{col.replace(" ", "_").lower()}_'
        blocks.append(pd.DataFrame(A_, columns=[f"{pref}{i}" for i in range(A_.shape[1])], index=df.index))
    return pd.concat(blocks, axis=1)

def assemble(base, ct_df, tfidf_df):
    X = pd.concat([base.reset_index(drop=True), ct_df.reset_index(drop=True),
                   tfidf_df.reset_index(drop=True)], axis=1)
    return X[[c for c in BASE_ORDER if c in X.columns] + [c for c in X.columns if c not in BASE_ORDER]]

def metrics(y_true, y_pred, name):
    a = np.abs((y_true - y_pred) / (y_true + 1e-10)) * 100
    return {"tập": name, "n": len(y_true), "R2": r2_score(y_true, y_pred),
            "MAPE": mean_absolute_percentage_error(y_true, y_pred) * 100,
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "APE<10%": float((a < 10).mean() * 100), "APE<15%": float((a < 15).mean() * 100)}

tr_idx, va_idx = train_test_split(np.arange(len(train_raw)), test_size=TEST_SIZE, random_state=SEED)
tr_raw, va_raw = train_raw.iloc[tr_idx], train_raw.iloc[va_idx]
y_tr, y_va = y_train_all[tr_idx], y_train_all[va_idx]

b_tr, medians = clean_base(tr_raw)
b_va, _ = clean_base(va_raw, medians); b_te, _ = clean_base(test_raw, medians)
ct_enc = ChiTietEncoder().fit(tr_raw[CHITIET])
yt_vecs = fit_tfidf(tr_raw)

X_tr = assemble(b_tr, ct_enc.transform(tr_raw[CHITIET]),  apply_tfidf(yt_vecs, tr_raw))
X_va = assemble(b_va, ct_enc.transform(va_raw[CHITIET]),  apply_tfidf(yt_vecs, va_raw))[X_tr.columns]
X_te = assemble(b_te, ct_enc.transform(test_raw[CHITIET]), apply_tfidf(yt_vecs, test_raw))[X_tr.columns]

FEATURES = list(X_tr.columns)
CATEGORIES = {c: list(pd.unique(X_tr[c])) for c in CAT_BASE}
assert not set(FEATURES) & set(EXCLUDED_ALL), "Có cột bị loại lọt vào đặc trưng"
assert not any(f.startswith("DG_") for f in FEATURES)

print(f"{len(FEATURES)} đặc trưng | train {X_tr.shape} · val {X_va.shape} · test {X_te.shape}")
print(f"  gốc: {len(CAT_BASE) + len(NUM_BASE)} · Chi tiết: {len(ct_enc.columns)} · "
      f"TF-IDF Yếu tố khác: {sum(f.startswith('tfidf_') for f in FEATURES)}")

646 đặc trưng | train (34772, 646) · val (8694, 646) · test (693, 646)
  gốc: 13 · Chi tiết: 33 · TF-IDF Yếu tố khác: 600


## 6. Huấn luyện LightGBM

In [6]:
def to_cat(X):
    return X.assign(**{c: pd.Categorical(X[c], categories=CATEGORIES[c]) for c in CAT_BASE})

ltr, lva = np.log1p(y_tr), np.log1p(y_va)

t0 = time.time()
lgb_model = lgb.LGBMRegressor(**LGB_PARAMS)
lgb_model.fit(to_cat(X_tr), ltr, eval_set=[(to_cat(X_va), lva)],
              categorical_feature=CAT_BASE,
              callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(1000)])
lgb_secs = round(time.time() - t0, 1)
BEST_ITER = int(lgb_model.best_iteration_)

PRED = {k: np.expm1(lgb_model.predict(to_cat(X))) for k, X in [("Train", X_tr), ("Val", X_va), ("Test", X_te)]}
Y    = {"Train": y_tr, "Val": y_va, "Test": y_test}
REG  = {"Train": tr_raw[VUNG].values, "Val": va_raw[VUNG].values, "Test": test_raw[VUNG].values}
print(f"\n=== LightGBM A1 === {lgb_secs}s | best_iter {BEST_ITER}")

[1000]	valid_0's l2: 0.0175957


[2000]	valid_0's l2: 0.0169854


[3000]	valid_0's l2: 0.0168743


[4000]	valid_0's l2: 0.0168425



=== LightGBM A1 === 99.3s | best_iter 3999


## 7. Đánh giá

### 7.1 Toàn bộ (như A)

In [7]:
ROUND = {"R2": 4, "MAPE": 2, "RMSE": 0, "APE<10%": 2, "APE<15%": 2}
met_all = pd.DataFrame([metrics(Y[k], PRED[k], k) for k in ["Train", "Val", "Test"]])
print(met_all.round(ROUND).to_string(index=False))

  tập     n     R2  MAPE       RMSE  APE<10%  APE<15%
Train 34772 0.9998  0.89  1213279.0    99.14    99.48
  Val  8694 0.9299  8.69 24208368.0    71.20    83.72
 Test   693 0.9492  9.51 17699783.0    64.07    79.08


### 7.2 Theo vùng HCM1 / HCM2

Cùng mô hình, cùng dự đoán, chỉ tách kết quả theo cột `Vùng`.

In [8]:
rows = []
for k in ["Train", "Val", "Test"]:
    for v in sorted(pd.unique(REG[k])):
        m = REG[k] == v
        rows.append({"vùng": v.upper(), **metrics(Y[k][m], PRED[k][m], k)})
met_reg = pd.DataFrame(rows)[["tập", "vùng", "n", "R2", "MAPE", "RMSE", "APE<10%", "APE<15%"]]
print(met_reg.round(ROUND).to_string(index=False))

print("\nPhân bố APE (%) trên Val và Test theo vùng:")
ape_rows = []
for k in ["Val", "Test"]:
    a = np.abs((Y[k] - PRED[k]) / Y[k]) * 100
    for v in sorted(pd.unique(REG[k])):
        d = pd.Series(a[REG[k] == v]).describe(percentiles=[.25, .5, .75, .9])
        ape_rows.append({"tập": k, "vùng": v.upper(), **d.drop("count").round(2).to_dict()})
print(pd.DataFrame(ape_rows).to_string(index=False))

  tập vùng     n     R2  MAPE       RMSE  APE<10%  APE<15%
Train HCM1 22294 0.9998  0.66  1389509.0    99.98   100.00
Train HCM2 12478 0.9990  1.30   807788.0    97.64    98.54
  Val HCM1  5575 0.9124  7.73 29545931.0    74.73    87.12
  Val HCM2  3119 0.9050 10.42  8555730.0    64.89    77.65
 Test HCM1   435 0.9348  8.67 21689872.0    67.82    82.53
 Test HCM2   258 0.9226 10.94  6949054.0    57.75    73.26

Phân bố APE (%) trên Val và Test theo vùng:
 tập vùng  mean   std  min  25%  50%   75%   90%    max
 Val HCM1  7.73  9.63 0.00 2.38 5.27 10.05 16.72 247.84
 Val HCM2 10.42 12.22 0.00 2.44 6.32 13.75 24.93 172.06
Test HCM1  8.67  8.77 0.01 2.46 6.02 12.23 19.24  55.51
Test HCM2 10.94 10.80 0.01 3.25 8.21 15.76 23.97  82.19


### 7.3 Kiểm định chênh lệch HCM1 – HCM2 trên tập test

Bootstrap khoảng tin cậy 95% cho hiệu MAPE (HCM2 − HCM1) và kiểm định Mann–Whitney trên APE (hai nhóm độc lập, nên không dùng kiểm định bắt cặp như A dùng khi so hai mô hình).

In [9]:
from scipy.stats import mannwhitneyu
for k in ["Val", "Test"]:
    a = np.abs((Y[k] - PRED[k]) / Y[k]) * 100
    a1, a2 = a[REG[k] == "hcm1"], a[REG[k] == "hcm2"]
    rng = np.random.default_rng(0)
    bs = np.array([a2[rng.integers(0, len(a2), len(a2))].mean() - a1[rng.integers(0, len(a1), len(a1))].mean()
                   for _ in range(20000)])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    p = mannwhitneyu(a2, a1).pvalue
    print(f"{k:5s} MAPE HCM1 {a1.mean():5.2f}% (n={len(a1)}) | HCM2 {a2.mean():5.2f}% (n={len(a2)}) | "
          f"Δ = {a2.mean() - a1.mean():+.2f} điểm  KTC 95% [{lo:+.2f}, {hi:+.2f}]  p = {p:.3f}  -> "
          f"{'khác biệt thật' if (lo > 0 or hi < 0) else 'KHÔNG phân biệt được với nhiễu'}")

Val   MAPE HCM1  7.73% (n=5575) | HCM2 10.42% (n=3119) | Δ = +2.70 điểm  KTC 95% [+2.20, +3.20]  p = 0.000  -> khác biệt thật


Test  MAPE HCM1  8.67% (n=435) | HCM2 10.94% (n=258) | Δ = +2.26 điểm  KTC 95% [+0.73, +3.83]  p = 0.004  -> khác biệt thật


### 7.4 Feature importance (theo GAIN)

Như A: dùng `gain`, không dùng số lần cắt (số lần cắt thiên vị cột số nhiều giá trị).

In [10]:
fi = pd.DataFrame({"Feature": FEATURES,
                   "Gain": lgb_model.booster_.feature_importance("gain"),
                   "So_lan_cat": lgb_model.booster_.feature_importance("split")})
fi["Gain_%"] = fi.Gain / fi.Gain.sum() * 100
fi = fi.sort_values("Gain", ascending=False).reset_index(drop=True)

print("Top 20 feature importance (theo GAIN):")
print(fi.head(20)[["Feature", "Gain_%", "So_lan_cat"]].round(3).to_string(index=False))
print()
for lab, mask in [("cột phân loại gốc", fi.Feature.isin(CAT_BASE)),
                  ("cột số gốc", fi.Feature.isin(NUM_BASE)),
                  ("Chi tiết (ct_*)", fi.Feature.str.startswith("ct_")),
                  ("TF-IDF Yếu tố khác", fi.Feature.str.startswith("tfidf_"))]:
    print(f"  {lab:22s} {fi[mask]['Gain_%'].sum():6.2f}% gain")

Top 20 feature importance (theo GAIN):
                                            Feature  Gain_%  So_lan_cat
                                               Vùng  39.482        2508
                             Xã/Phường/Thị trấn mới  34.517       43185
Độ rộng ngõ/ngách nhỏ nhất (Từ đường chính đến BĐS)   6.699      133335
                                          Đường phố   6.490       46110
                                     Diện tích (m2)   1.975      169455
                               Kích thước chiều dài   1.668      122934
                                 Lợi thế kinh doanh   1.322       26620
                    Khoảng cách đến đường chính (m)   1.045       48408
                            Kích thước mặt tiền (m)   0.644       84197
                                           ct_emb_0   0.485       16948
                                           ct_emb_4   0.441        9112
                             Vị trí trong khung giá   0.432        3634
                         

## 8. Đóng gói mô hình

In [11]:
def sha1(path, buf=1 << 20):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()

os.makedirs(MODEL_DIR, exist_ok=True)
lgb_model.booster_.save_model(str(MODEL_DIR / "lightgbm_model.txt"))

VUNG_FROM_XA = (train_raw.groupby(train_raw[XA].astype(str))[VUNG]
                .agg(lambda s: s.mode().iat[0] if len(s.mode()) else "Unknown").to_dict())
VUNG_MODE = train_raw[VUNG].mode().iat[0]

joblib.dump({"schema_version": 1, "version": VERSION, "region": "HCM", "model": "LightGBM",
             "chitiet": ct_enc.get_state(), "yeu_to_khac": yt_vecs, "medians": medians,
             "features": FEATURES, "cat_cols": CAT_BASE, "categories": CATEGORIES,
             "vung_from_xa": VUNG_FROM_XA, "vung_mode": VUNG_MODE,
             "cols": {"target": TARGET, "cat_base": CAT_BASE, "num_base": NUM_BASE,
                      "base_order": BASE_ORDER, "text_tfidf": TEXT_TFIDF, "chitiet": CHITIET,
                      "xa": XA, "vung": VUNG},
             "excluded_cols": EXCLUDED, "target_transform": "log1p"},
            MODEL_DIR / "preprocessor.joblib", compress=3)

manifest = {"name": VERSION, "base_version": "A (A/pipeline_HCM_A.ipynb)",
            "region": "Thành phố Hồ Chí Minh", "model": "LightGBM",
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "target": TARGET, "target_transform": "log1p", "n_features": len(FEATURES),
            "cat_features": CAT_BASE, "num_features": NUM_BASE, "excluded_cols": EXCLUDED,
            "lightgbm_params": LGB_PARAMS, "early_stopping": EARLY_STOP,
            "split": {"test_size": TEST_SIZE, "random_state": SEED, "n_train": int(len(X_tr)),
                      "n_val": int(len(X_va)), "n_test": int(len(X_te))},
            "results": {"secs": lgb_secs, "best_iter": BEST_ITER,
                        "metrics": met_all.to_dict("records"),
                        "metrics_by_region": met_reg.to_dict("records")},
            "sources": {p.name: {"sha1": sha1(p), "rows": int(n)}
                        for p, n in [(TRAIN_XLS, len(train_raw)), (TEST_XLS, len(test_raw))]},
            "env": {"python": platform.python_version(), "pandas": pd.__version__,
                    "numpy": np.__version__, "sklearn": sklearn.__version__,
                    "lightgbm": lgb.__version__}}
with open(MODEL_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=float)

print("Đã đóng gói vào", MODEL_DIR)
for fn in sorted(os.listdir(MODEL_DIR)):
    print(f"  {fn:24s} {os.path.getsize(MODEL_DIR / fn) / 1024:9,.0f} KB")

Đã đóng gói vào D:\HCM2810\A1\model_HCM_A1
  lightgbm_model.txt         107,800 KB
  manifest.json                    5 KB
  preprocessor.joblib            354 KB


## 9. Nạp lại & dự đoán trên dữ liệu mới

In [12]:
def load_bundle(model_dir=MODEL_DIR):
    st = joblib.load(Path(model_dir) / "preprocessor.joblib")
    return {"lightgbm": lgb.Booster(model_file=str(Path(model_dir) / "lightgbm_model.txt")),
            "ct_enc": ChiTietEncoder.from_state(st["chitiet"]), "yt_vecs": st["yeu_to_khac"],
            "medians": st["medians"], "features": st["features"],
            "cat_cols": st["cat_cols"], "categories": st["categories"],
            "vung_from_xa": st["vung_from_xa"], "vung_mode": st["vung_mode"]}

def predict_new(df, bundle=None):
    """df: dữ liệu ĐÃ LÀM SẠCH theo cùng quy trình output/ (cùng tên cột)."""
    b = bundle if bundle is not None else load_bundle()
    df = df.reset_index(drop=True).copy()
    if VUNG not in df.columns or df[VUNG].isna().all():          # thiếu Vùng thì tra từ xã/phường
        df[VUNG] = df[XA].astype(str).map(b["vung_from_xa"]).fillna(b["vung_mode"])
        print(f"  [predict_new] thiếu '{VUNG}' -> đã tra lại từ '{XA}'")
    base, _ = clean_base(df, b["medians"])
    X = assemble(base, b["ct_enc"].transform(df[CHITIET]), apply_tfidf(b["yt_vecs"], df))
    for c in b["features"]:
        if c not in X.columns:
            X[c] = 0.0
    X = X[b["features"]]
    X = X.assign(**{c: pd.Categorical(X[c].astype(str), categories=b["categories"][c]) for c in b["cat_cols"]})
    return np.expm1(b["lightgbm"].predict(X))

bundle = load_bundle()
p_reload = predict_new(test_raw, bundle)
print(f"Nạp lại — sai lệch tối đa so với mô hình trong bộ nhớ: {np.abs(p_reload - PRED['Test']).max():.6e}")
assert np.allclose(p_reload, PRED["Test"], rtol=1e-9, atol=1e-6)
print("=> Gói mô hình nạp lại khớp tuyệt đối.")

_drop = [c for c in ["Thành phố/Quận/Huyện/Thị xã", VUNG] + EXCLUDED_ALL if c in test_raw.columns]
p_prod = predict_new(test_raw.drop(columns=_drop), bundle)
print(f"\nBỏ hết cột không dùng + '{VUNG}' khỏi file đầu vào: chênh lệch tối đa {np.abs(p_prod - p_reload).max():.3e}")
print(f"  MAPE: {np.abs((y_test - p_prod) / y_test).mean() * 100:.2f}% "
      f"(bản đầy đủ {np.abs((y_test - p_reload) / y_test).mean() * 100:.2f}%)")

Nạp lại — sai lệch tối đa so với mô hình trong bộ nhớ: 0.000000e+00
=> Gói mô hình nạp lại khớp tuyệt đối.
  [predict_new] thiếu 'Vùng' -> đã tra lại từ 'Xã/Phường/Thị trấn mới'



Bỏ hết cột không dùng + 'Vùng' khỏi file đầu vào: chênh lệch tối đa 0.000e+00
  MAPE: 9.51% (bản đầy đủ 9.51%)


## 10. Xuất kết quả

In [13]:
out = test_raw.copy()
out["Predicted_Đơn giá"] = PRED["Test"]
out["Actual_Đơn giá"] = y_test
out["APE_%"] = np.abs((y_test - out["Predicted_Đơn giá"]) / (y_test + 1e-10)) * 100
out.to_excel(RES_DIR / "test_predictions_LightGBM.xlsx", index=False)

val_out = va_raw.copy()
val_out["Predicted_Đơn giá"] = PRED["Val"]
val_out["APE_%"] = np.abs((y_va - PRED["Val"]) / y_va) * 100
val_out.to_excel(RES_DIR / "val_predictions_LightGBM.xlsx", index=False)

with pd.ExcelWriter(RES_DIR / "ket_qua_danh_gia_LightGBM.xlsx") as w:
    met_all.round(ROUND).to_excel(w, sheet_name="tong", index=False)
    met_reg.round(ROUND).to_excel(w, sheet_name="theo_vung", index=False)
fi.to_csv(RES_DIR / "feature_importance_LightGBM.csv", index=False, encoding="utf-8-sig")

print("Đã lưu vào", RES_DIR)
for fn in sorted(os.listdir(RES_DIR)):
    print("  ", fn)
print(f"\nA1 · LightGBM · {len(FEATURES)} đặc trưng · best_iter {BEST_ITER}")
print(met_all.round(ROUND).to_string(index=False))
print()
print(met_reg[met_reg["tập"] == "Test"].round(ROUND).to_string(index=False))

Đã lưu vào D:\HCM2810\A1\ket_qua
   feature_importance_HCM_A1.csv
   ket_qua_danh_gia_HCM_A1.xlsx
   test_predictions_HCM_A1.xlsx
   val_predictions_HCM_A1.xlsx

A1 · LightGBM · 646 đặc trưng · best_iter 3999
  tập     n     R2  MAPE       RMSE  APE<10%  APE<15%
Train 34772 0.9998  0.89  1213279.0    99.14    99.48
  Val  8694 0.9299  8.69 24208368.0    71.20    83.72
 Test   693 0.9492  9.51 17699783.0    64.07    79.08

 tập vùng   n     R2  MAPE       RMSE  APE<10%  APE<15%
Test HCM1 435 0.9348  8.67 21689872.0    67.82    82.53
Test HCM2 258 0.9226 10.94  6949054.0    57.75    73.26
